In [1]:
import os, sys

# Repository information
REPO_NAME = "RecSys-Challenge-2025"
REPO_URL  = f"github.com/Lv1g1/{REPO_NAME}.git"

# Detect environment
IS_COLAB = 'content' in os.getcwd()
IS_KAGGLE = 'kaggle' in os.getcwd()
IS_LOCAL = not (IS_COLAB or IS_KAGGLE)

if IS_COLAB:
    WORKING_DIR = "/content"

    # Mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

    # Get GitHub token via input
    def get_token():
        from getpass import getpass
        return getpass("GitHub Token: ")

elif IS_KAGGLE:
    WORKING_DIR = "/kaggle/working"

    # Get GitHub token from Kaggle secrets
    def get_token():
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("Token")

# If local environment assume inside the repo
LOCAL_REPO_PATH = os.getcwd() if IS_LOCAL else os.path.join(WORKING_DIR, REPO_NAME)

# Clone the repository if it doesn't exist
if not os.path.exists(LOCAL_REPO_PATH):
    os.chdir(WORKING_DIR)
    token = get_token()

    !git clone https://{token}@{REPO_URL}
else:
    print("Repo already exists — pulling latest changes")
    os.chdir(LOCAL_REPO_PATH)
    !git pull
    os.chdir(WORKING_DIR)

# Add to Python PATH
if LOCAL_REPO_PATH not in sys.path:
    sys.path.append(LOCAL_REPO_PATH)

Repo already exists — pulling latest changes
Already up to date.


In [2]:
if False:  # Set to True if you want to recompile Cython files
    os.chdir(LOCAL_REPO_PATH)
    !python run_compile_all_cython.py
    os.chdir(WORKING_DIR)

In [3]:
%%capture
if not IS_LOCAL:
    !pip install optuna

import optuna

In [4]:
import importlib
import scipy.sparse as sps
import pandas as pd
import numpy as np

from Challenge import paths
importlib.reload(paths)

from Challenge.hyper_tuning import hyperparameter_tuning

Running on kaggle — storage at: /kaggle/working
Running on kaggle — storage at: /kaggle/working


In [5]:
# Load datasets
URM_train = sps.load_npz(paths.URM_TRAIN)
URM_validation = sps.load_npz(paths.URM_VALIDATION)

In [6]:
def evaluate_recommender(recommender, at):
    cumulative_recall = 0.0
    num_eval = 0
    
    for user_id in range(URM_validation.shape[0]):
        relevant_items = URM_validation.indices[URM_validation.indptr[user_id]:URM_validation.indptr[user_id+1]]
        
        if len(relevant_items)>0:
            num_eval+=1
            
            recommended_items = recommender.recommend(user_id, cutoff=at)
            
            is_relevant = np.isin(recommended_items, relevant_items, assume_unique=True)
            recall_score = np.sum(is_relevant, dtype=np.float32) / relevant_items.shape[0]

            cumulative_recall += recall_score

    return cumulative_recall / num_eval

# Train a KNN for each similarity

["cosine", "pearson", "jaccard", "tversky"]

In [7]:
from Recommenders.KNN.UserKNNCFRecommender import UserKNNCFRecommender

In [8]:
def perform_optimization(similarity, n_trials):
    # Define objective function for hyperparameter tuning
    STUDY_NAME = UserKNNCFRecommender.RECOMMENDER_NAME + "_" + similarity
    
    def objective_function(optuna_trial: optuna.trial.Trial) -> float:
        recommender_instance = UserKNNCFRecommender(URM_train)
        recommender_instance.fit(
            similarity=similarity,
            topK=optuna_trial.suggest_int("topK", 10, 1500),
            shrink=optuna_trial.suggest_int("shrink", 0, 2000),
            normalize=optuna_trial.suggest_categorical("normalize", [True, False]),
            feature_weighting=optuna_trial.suggest_categorical("feature_weighting", ["BM25", "TF-IDF", "none"])
        )
    
        return evaluate_recommender(recommender_instance, at=20)

    # Perform hyperparameter tuning
    save_results, optuna_study = hyperparameter_tuning(
        objective_function,
        study_name=STUDY_NAME,
        n_trials=n_trials
    )

    return save_results, optuna_study

## Cosine

In [80]:
SIMILARITY = "cosine"

In [64]:
fd_results, optuna_study = perform_optimization(SIMILARITY, 100)

  0%|          | 0/100 [00:00<?, ?it/s]

Similarity column 27095 (100.0%), 707.95 column/sec. Elapsed time 38.27 sec
[I 2025-11-08 18:10:09,958] Trial 0 finished with value: 0.2001221301441264 and parameters: {'topK': 1254, 'shrink': 80, 'normalize': True, 'feature_weighting': 'none'}. Best is trial 0 with value: 0.2001221301441264.
Similarity column 27095 (100.0%), 752.41 column/sec. Elapsed time 36.01 sec
[I 2025-11-08 18:11:00,552] Trial 1 finished with value: 0.06876554509381957 and parameters: {'topK': 83, 'shrink': 1549, 'normalize': True, 'feature_weighting': 'BM25'}. Best is trial 0 with value: 0.2001221301441264.
Similarity column 27095 (100.0%), 728.21 column/sec. Elapsed time 37.21 sec
[I 2025-11-08 18:12:08,157] Trial 2 finished with value: 0.21741736932904626 and parameters: {'topK': 1090, 'shrink': 1020, 'normalize': True, 'feature_weighting': 'TF-IDF'}. Best is trial 2 with value: 0.21741736932904626.
Similarity column 27095 (100.0%), 738.51 column/sec. Elapsed time 36.69 sec
[I 2025-11-08 18:13:08,938] Trial 3

In [65]:
optuna.visualization.plot_optimization_history(optuna_study)

In [66]:
optuna.visualization.plot_param_importances(optuna_study)

In [67]:
optuna.visualization.plot_parallel_coordinate(optuna_study)

In [81]:
STUDY_NAME = UserKNNCFRecommender.RECOMMENDER_NAME + "_tuning_" + SIMILARITY

def tversky_tuning_function(optuna_trial: optuna.trial.Trial) -> float:
    recommender_instance = UserKNNCFRecommender(URM_train)
    recommender_instance.fit(
        similarity=SIMILARITY,
        topK=optuna_trial.suggest_int("topK", 10, 500),
        shrink=optuna_trial.suggest_int("shrink", 0, 100),
        normalize=True,
        feature_weighting="TF-IDF"
    )

    return evaluate_recommender(recommender_instance, at=20)

# Perform hyperparameter tuning
save_results, optuna_study = hyperparameter_tuning(
    tversky_tuning_function,
    study_name=STUDY_NAME,
    n_trials=40
)

  0%|          | 0/40 [00:00<?, ?it/s]

Similarity column 27095 (100.0%), 732.52 column/sec. Elapsed time 36.99 sec
[I 2025-11-08 19:58:19,333] Trial 0 finished with value: 0.22321767017582314 and parameters: {'topK': 379, 'shrink': 33}. Best is trial 0 with value: 0.22321767017582314.
Similarity column 27095 (100.0%), 734.65 column/sec. Elapsed time 36.88 sec
[I 2025-11-08 19:59:14,821] Trial 1 finished with value: 0.2185996839917792 and parameters: {'topK': 195, 'shrink': 89}. Best is trial 0 with value: 0.22321767017582314.
Similarity column 27095 (100.0%), 735.32 column/sec. Elapsed time 36.85 sec
[I 2025-11-08 20:00:10,278] Trial 2 finished with value: 0.21960173727314483 and parameters: {'topK': 204, 'shrink': 48}. Best is trial 0 with value: 0.22321767017582314.
Similarity column 27095 (100.0%), 729.97 column/sec. Elapsed time 37.12 sec
[I 2025-11-08 20:01:07,674] Trial 3 finished with value: 0.2220390272977302 and parameters: {'topK': 280, 'shrink': 25}. Best is trial 0 with value: 0.22321767017582314.
Similarity col

In [82]:
optuna.visualization.plot_optimization_history(optuna_study)

### Best Mode
- Best Value: 0.22335921189714308
- Best Params: {'topK': 347, 'shrink': 3, 'normalize': True, 'feature_weighting': 'TF-IDF'}

## Tversky

In [9]:
SIMILARITY = "tversky"

In [10]:
fd_results, optuna_study = perform_optimization(SIMILARITY, 10)

  0%|          | 0/10 [00:00<?, ?it/s]

Similarity column 27095 (100.0%), 664.61 column/sec. Elapsed time 40.77 sec
[I 2025-11-08 22:24:39,133] Trial 97 finished with value: 0.10582713390339181 and parameters: {'topK': 317, 'shrink': 1739, 'normalize': False, 'feature_weighting': 'TF-IDF'}. Best is trial 76 with value: 0.22106383032143062.
Similarity column 27095 (100.0%), 687.09 column/sec. Elapsed time 39.43 sec
[I 2025-11-08 22:25:42,184] Trial 98 finished with value: 0.21672065234318236 and parameters: {'topK': 445, 'shrink': 48, 'normalize': False, 'feature_weighting': 'TF-IDF'}. Best is trial 76 with value: 0.22106383032143062.
Similarity column 27095 (100.0%), 659.69 column/sec. Elapsed time 41.07 sec
[I 2025-11-08 22:26:50,296] Trial 99 finished with value: 0.19988375331771902 and parameters: {'topK': 499, 'shrink': 148, 'normalize': False, 'feature_weighting': 'TF-IDF'}. Best is trial 76 with value: 0.22106383032143062.
Similarity column 27095 (100.0%), 670.65 column/sec. Elapsed time 40.40 sec
[I 2025-11-08 22:27:5

In [11]:
optuna.visualization.plot_optimization_history(optuna_study)

In [12]:
optuna.visualization.plot_param_importances(optuna_study)

In [13]:
optuna.visualization.plot_parallel_coordinate(optuna_study)

In [14]:
STUDY_NAME = UserKNNCFRecommender.RECOMMENDER_NAME + "_tuning_" + SIMILARITY

def tversky_tuning_function(optuna_trial: optuna.trial.Trial) -> float:
    recommender_instance = UserKNNCFRecommender(URM_train)
    recommender_instance.fit(
        similarity=SIMILARITY,
        topK=optuna_trial.suggest_int("topK", 400, 500),
        shrink=optuna_trial.suggest_int("shrink", 0, 20),
        tversky_alpha=optuna_trial.suggest_float("tversky_alpha", 0., 1.),
        tversky_beta=optuna_trial.suggest_float("tversky_beta", 0., 1.),
        normalize=False,
        feature_weighting="TF-IDF"
    )

    return evaluate_recommender(recommender_instance, at=20)

# Perform hyperparameter tuning
save_results, optuna_study = hyperparameter_tuning(
    tversky_tuning_function,
    study_name=STUDY_NAME,
    n_trials=40
)

  0%|          | 0/40 [00:00<?, ?it/s]

Similarity column 27095 (100.0%), 677.01 column/sec. Elapsed time 40.02 sec
[I 2025-11-08 22:37:37,628] Trial 0 finished with value: 0.2133319233622239 and parameters: {'topK': 491, 'shrink': 10, 'tversky_alpha': 0.922685086560159, 'tversky_beta': 0.3865348734506876}. Best is trial 0 with value: 0.2133319233622239.
Similarity column 27095 (100.0%), 676.38 column/sec. Elapsed time 40.06 sec
[I 2025-11-08 22:38:42,627] Trial 1 finished with value: 0.21991936647412047 and parameters: {'topK': 473, 'shrink': 10, 'tversky_alpha': 0.7321294993436386, 'tversky_beta': 0.6901387068861079}. Best is trial 1 with value: 0.21991936647412047.
Similarity column 27095 (100.0%), 651.46 column/sec. Elapsed time 41.59 sec
[I 2025-11-08 22:39:51,328] Trial 2 finished with value: 0.2196891296710707 and parameters: {'topK': 499, 'shrink': 2, 'tversky_alpha': 0.06278380287277485, 'tversky_beta': 0.3812204576801994}. Best is trial 1 with value: 0.21991936647412047.
Similarity column 27095 (100.0%), 665.28 col

In [15]:
optuna.visualization.plot_optimization_history(optuna_study)

In [16]:
optuna.visualization.plot_param_importances(optuna_study)

In [17]:
optuna.visualization.plot_parallel_coordinate(optuna_study)

### **Best Model**
- Best Value: 0.22106383032143062
- Best Params: {'topK': 438, 'shrink': 3, 'normalize': False, 'feature_weighting': 'TF-IDF'}

- Best Value: 0.22316709140849905
- Best Params: {'topK': 401, 'shrink': 5, 'tversky_alpha': 0.2184775930703794, 'tversky_beta': 0.5934354684918872}